# NASA POWER pairwise classification delta

In [ ]:
from __future__ import annotations

import math
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from shapely.geometry import Point

from liljegren_analysis.country.cells import (
    cached_natural_earth_boundaries,
    load_countries,
)

In [ ]:
# Analysis parameters: paths are resolved relative to the repository root.
RUN_DIRECTORY = Path(
    ".hsb/runs/wbgt-comparison-20260827-144607"
)
SHARD_ORDINAL = 1
CYCLE_ORDINAL = 1
ANALYSIS_START_DATE = date(2001, 1, 1)
ANALYSIS_END_DATE = date(2025, 12, 31)
COUNTRY_CODES: list[str] | None = ["SG"]  # ISO alpha-2/alpha-3; e.g. ["SG"]
LOCATION_IDS: list[str] | None = ["singapore"]  # Example: ["singapore", "darwin"]
WBGT_BAND_PROFILES = {
    "sg_moh_public": [31, 33],
    "sg_mom_outdoor_controls": [31, 32, 33],
    "sg_wsh_workplace_risk": [31, 32],
}
WBGT_THRESHOLDS_C = WBGT_BAND_PROFILES["sg_mom_outdoor_controls"]

In [ ]:
working_directory = Path.cwd().resolve()
repository = next(
    (
        candidate
        for candidate in (working_directory, *working_directory.parents)
        if (candidate / "pyproject.toml").is_file()
    ),
    None,
)
if repository is None:
    raise FileNotFoundError("Could not locate the repository root")

run_directory = RUN_DIRECTORY if RUN_DIRECTORY.is_absolute() else repository / RUN_DIRECTORY
workload_path = run_directory / "dataset" / "workload.csv"
results_directory = run_directory / "results"
start_date = ANALYSIS_START_DATE
end_date = ANALYSIS_END_DATE
if start_date > end_date:
    raise ValueError("ANALYSIS_START_DATE must not be after ANALYSIS_END_DATE")
thresholds_c = tuple(float(value) for value in WBGT_THRESHOLDS_C)
if (
    not thresholds_c
    or any(not math.isfinite(value) for value in thresholds_c)
    or any(left >= right for left, right in zip(thresholds_c, thresholds_c[1:]))
):
    raise ValueError("WBGT_THRESHOLDS_C must be finite and strictly increasing")
if not workload_path.is_file():
    raise FileNotFoundError(f"NASA POWER workload does not exist: {workload_path}")

result_paths = {
    (int(path.parents[2].name), path.parent.name): path
    for path in sorted(
        results_directory.glob(f"*/{int(SHARD_ORDINAL):04d}/*/results.parquet")
    )
    if path.parents[2].name.isdigit()
    and start_date.year <= int(path.parents[2].name) <= end_date.year
}
result_partitions = tuple(sorted({partition for partition, _ in result_paths}))
implementations = tuple(sorted({implementation for _, implementation in result_paths}))
missing_results = {
    (partition, implementation)
    for partition in result_partitions
    for implementation in implementations
} - set(result_paths)
if len(implementations) < 2 or not result_partitions:
    raise FileNotFoundError(
        f"Expected at least two targets and one result partition beneath {results_directory}"
    )
if missing_results:
    raise FileNotFoundError(f"Missing partition/target results: {sorted(missing_results)}")

{
    "run_directory": run_directory,
    "shard_ordinal": SHARD_ORDINAL,
    "cycle_ordinal": CYCLE_ORDINAL,
    "date_range": (start_date, end_date),
    "result_partitions": result_partitions,
    "targets": implementations,
    "thresholds_c": thresholds_c,
}

## Load workload metadata and target results

In [ ]:
workload_columns = [
    "row_id",
    "location_id",
    "interval_start_utc",
    "longitude_deg_east",
    "latitude_deg_north",
]
workload = pd.read_csv(
    workload_path,
    usecols=workload_columns,
    parse_dates=["interval_start_utc"],
).rename(columns={"row_id": "source_row_id"})
workload.insert(0, "result_partition", workload["interval_start_utc"].dt.year)
workload.insert(1, "row_ordinal", (
    workload.groupby("result_partition", sort=False).cumcount() + 1
))
if workload["source_row_id"].duplicated().any():
    raise ValueError("workload.csv contains duplicate row_id values")

all_locations = sorted(workload["location_id"].unique())
selected_locations = None if LOCATION_IDS is None else list(
    dict.fromkeys(value.strip().lower() for value in LOCATION_IDS)
)
unknown_locations = sorted(set(selected_locations or ()) - set(all_locations))
if unknown_locations:
    raise ValueError(f"Unknown LOCATION_IDS: {unknown_locations}")

selected_countries = []
if COUNTRY_CODES is not None:
    requested_country_codes = list(
        dict.fromkeys(value.strip().upper() for value in COUNTRY_CODES)
    )
    if not requested_country_codes:
        raise ValueError("COUNTRY_CODES must be None or contain at least one code")
    boundaries = cached_natural_earth_boundaries(repository / "data" / "boundaries")
    selected_countries = load_countries(boundaries, requested_country_codes)

    location_coordinates = workload[
        ["location_id", "longitude_deg_east", "latitude_deg_north"]
    ].drop_duplicates()
    duplicate_coordinates = location_coordinates["location_id"].duplicated(keep=False)
    if duplicate_coordinates.any():
        duplicate_locations = sorted(
            location_coordinates.loc[duplicate_coordinates, "location_id"].unique()
        )
        raise ValueError(f"Locations have multiple coordinates: {duplicate_locations}")

    location_country_records = []
    for row in location_coordinates.itertuples(index=False):
        point = Point(row.longitude_deg_east, row.latitude_deg_north)
        matches = [
            country for country in selected_countries if country.geometry.covers(point)
        ]
        if len(matches) > 1:
            raise ValueError(
                f"Location {row.location_id!r} matches multiple selected countries: "
                f"{[country.iso_a3 for country in matches]}"
            )
        if matches:
            country = matches[0]
            location_country_records.append(
                {
                    "location_id": row.location_id,
                    "country_iso_a2": country.iso_a2,
                    "country_iso_a3": country.iso_a3,
                    "country_name": country.name,
                }
            )
    location_countries = pd.DataFrame(
        location_country_records,
        columns=[
            "location_id", "country_iso_a2", "country_iso_a3", "country_name"
        ],
    )
    workload = workload.merge(
        location_countries, on="location_id", how="inner", validate="many_to_one"
    )

if selected_locations is not None:
    workload = workload.loc[workload["location_id"].isin(selected_locations)].copy()
analysis_start = pd.Timestamp(start_date, tz="UTC")
analysis_end_exclusive = pd.Timestamp(end_date, tz="UTC") + pd.Timedelta(days=1)
workload = workload.loc[
    workload["interval_start_utc"].ge(analysis_start)
    & workload["interval_start_utc"].lt(analysis_end_exclusive)
].copy()
if workload.empty:
    raise ValueError("No workload rows match the date, country, and location filters")

country_scope = (
    "-".join(country.iso_a3.lower() for country in selected_countries)
    if selected_countries
    else "all"
)
output_directory = (
    repository / "data" / "country_classification" / country_scope
)
country_filter = [
    (country.iso_a2, country.iso_a3, country.name) for country in selected_countries
]
display(
    pd.DataFrame(
        {
            "country_filter": [country_filter or None],
            "selected_locations": [workload["location_id"].nunique()],
            "workload_rows": [len(workload)],
            "first_timestamp": [workload["interval_start_utc"].min()],
            "last_timestamp": [workload["interval_start_utc"].max()],
            "output_directory": [output_directory],
        }
    )
)
location_summary_columns = [
    *(
        ["country_iso_a2", "country_iso_a3", "country_name"]
        if COUNTRY_CODES is not None
        else []
    ),
    "location_id",
]
display(workload.groupby(location_summary_columns, as_index=False).size())

In [ ]:
required_result_columns = {
    "cycle_ordinal", "row_id", "row_ordinal", "wbgt_status", "wbgt_c"
}
selected_ordinals_by_partition = {
    int(partition): pd.Index(frame["row_ordinal"])
    for partition, frame in workload.groupby("result_partition", sort=False)
}
result_frames = []
for (result_partition, implementation), result_path in result_paths.items():
    selected_ordinals = selected_ordinals_by_partition.get(result_partition)
    if selected_ordinals is None:
        continue
    result = pd.read_parquet(result_path, columns=sorted(required_result_columns))
    missing_columns = required_result_columns - set(result.columns)
    if missing_columns:
        raise ValueError(f"{result_path} is missing columns: {sorted(missing_columns)}")
    result = result.loc[result["cycle_ordinal"] == CYCLE_ORDINAL].copy()
    if result["row_ordinal"].duplicated().any():
        raise ValueError(f"{result_path} contains duplicate row_ordinal values")
    result = result.loc[result["row_ordinal"].isin(selected_ordinals)].copy()
    result.insert(0, "result_partition", result_partition)
    result.insert(0, "implementation", implementation)
    result_frames.append(result)

results = pd.concat(result_frames, ignore_index=True)
result_row_id_counts = results.groupby(
    ["result_partition", "cycle_ordinal", "row_ordinal"]
)["row_id"].nunique()
if result_row_id_counts.gt(1).any():
    raise ValueError("Targets disagree on result row_id for the same row_ordinal")
wbgt_status_summary = (
    results.groupby(["implementation", "wbgt_status"], as_index=False, sort=True)
    .size()
    .rename(columns={"size": "n"})
)
observations = (
    results.loc[results["wbgt_status"].eq("ok") & results["wbgt_c"].notna()]
    .rename(columns={"row_id": "result_row_id"})
    .merge(
        workload,
        on=["result_partition", "row_ordinal"],
        how="inner",
        validate="many_to_one",
    )
)
if observations.empty:
    raise ValueError("No successful observations remain after filtering")

display(wbgt_status_summary)
display(
    observations.groupby("implementation", as_index=False).agg(
        valid_wbgt_observations=("wbgt_c", "size"),
        locations=("location_id", "nunique"),
        minimum_wbgt_c=("wbgt_c", "min"),
        maximum_wbgt_c=("wbgt_c", "max"),
    )
)

## Classify and pair

In [ ]:
bounds = (-np.inf, *thresholds_c, np.inf)
class_labels = tuple(
    (
        f"< {upper:g} C"
        if index == 0
        else f">= {lower:g} C"
        if index == len(thresholds_c)
        else f"[{lower:g}, {upper:g}) C"
    )
    for index, (lower, upper) in enumerate(zip(bounds, bounds[1:]))
)
class_definitions = pd.DataFrame(
    {"class_id": range(len(class_labels)), "class_label": class_labels}
)
class_label_by_id = dict(
    zip(class_definitions["class_id"], class_definitions["class_label"])
)

classified = observations.copy()
classified["class_id"] = pd.cut(
    classified["wbgt_c"], bins=bounds, labels=False, right=False
).astype("int8")
classified["class_label"] = classified["class_id"].map(class_label_by_id)
display(class_definitions)
display(classified.head(20))

In [ ]:
pair_keys = [
    "result_partition",
    "cycle_ordinal",
    "row_ordinal",
    "result_row_id",
    "source_row_id",
    "location_id",
    "interval_start_utc",
    "longitude_deg_east",
    "latitude_deg_north",
]
if COUNTRY_CODES is not None:
    pair_keys += ["country_iso_a2", "country_iso_a3", "country_name"]
pair_values = pair_keys + ["implementation", "wbgt_c", "class_id"]
paired_classifications = classified[pair_values].merge(
    classified[pair_values],
    on=pair_keys,
    how="inner",
    suffixes=("_a", "_b"),
    validate="many_to_many",
)
paired_classifications = paired_classifications.loc[
    paired_classifications["implementation_a"]
    < paired_classifications["implementation_b"]
].copy()
paired_classifications["wbgt_difference_c"] = (
    paired_classifications["wbgt_c_a"]
    - paired_classifications["wbgt_c_b"]
)
paired_classifications["class_difference"] = (
    paired_classifications["class_id_a"]
    - paired_classifications["class_id_b"]
)
paired_classifications["classification_changed"] = (
    paired_classifications["class_difference"] != 0
)
paired_classifications["result_month"] = (
    paired_classifications["interval_start_utc"].dt.strftime("%Y-%m")
)
paired_classifications.reset_index(drop=True, inplace=True)
paired_classifications.info()
display(paired_classifications.head(20))

## Classification-change summaries

In [ ]:
pair_columns = ["implementation_a", "implementation_b"]


def summarize_changes(frame: pd.DataFrame, group_columns: list[str]) -> pd.DataFrame:
    summary = (
        frame.assign(
            a_higher_class=frame["class_difference"] > 0,
            b_higher_class=frame["class_difference"] < 0,
        )
        .groupby(group_columns, as_index=False, sort=True)
        .agg(
            n_paired=("classification_changed", "size"),
            n_changed=("classification_changed", "sum"),
            a_higher_class=("a_higher_class", "sum"),
            b_higher_class=("b_higher_class", "sum"),
        )
    )
    summary["percent_changed"] = (
        100.0 * summary["n_changed"] / summary["n_paired"]
    ).round(4)
    return summary[
        group_columns
        + [
            "n_paired",
            "n_changed",
            "percent_changed",
            "a_higher_class",
            "b_higher_class",
        ]
    ]


change_summary = summarize_changes(paired_classifications, pair_columns)
location_change_summary = summarize_changes(
    paired_classifications, ["location_id", *pair_columns]
)
monthly_change_summary = summarize_changes(
    paired_classifications, ["result_month", *pair_columns]
)
country_change_summary = (
    summarize_changes(paired_classifications, ["country_iso_a3", *pair_columns])
    if COUNTRY_CODES is not None
    else None
)
display(change_summary)
if country_change_summary is not None:
    display(country_change_summary)
display(location_change_summary)
display(monthly_change_summary)

In [ ]:
def count_transitions(frame: pd.DataFrame, group_columns: list[str]) -> pd.DataFrame:
    transition_columns = group_columns + ["class_id_a", "class_id_b"]
    counts = (
        frame.groupby(transition_columns, as_index=False, sort=True)
        .size()
        .rename(columns={"size": "n"})
    )
    counts["class_a_label"] = counts["class_id_a"].map(class_label_by_id)
    counts["class_b_label"] = counts["class_id_b"].map(class_label_by_id)
    counts["classification_changed"] = counts["class_id_a"] != counts["class_id_b"]
    return counts


transition_counts = count_transitions(paired_classifications, pair_columns)
location_transition_counts = count_transitions(
    paired_classifications, ["location_id", *pair_columns]
)
monthly_transition_counts = count_transitions(
    paired_classifications, ["result_month", *pair_columns]
)
country_transition_counts = (
    count_transitions(paired_classifications, ["country_iso_a3", *pair_columns])
    if COUNTRY_CODES is not None
    else None
)
display(transition_counts)

In [ ]:
changed_observations = (
    paired_classifications.loc[paired_classifications["classification_changed"]]
    .assign(
        absolute_wbgt_difference_c=lambda frame: frame["wbgt_difference_c"].abs()
    )
    .sort_values(
        ["absolute_wbgt_difference_c", "interval_start_utc", "source_row_id"],
        ascending=[False, True, True],
    )
    .head(100)
)
display(changed_observations)

## Export compact classification results

In [ ]:
outputs = {
    "classes.csv": class_definitions,
    "pair_change_summary.csv": change_summary,
    "location_pair_change_summary.csv": location_change_summary,
    "monthly_pair_change_summary.csv": monthly_change_summary,
    "pair_transition_counts.csv": transition_counts,
    "location_pair_transition_counts.csv": location_transition_counts,
    "monthly_pair_transition_counts.csv": monthly_transition_counts,
    "top_classification_changes.csv": changed_observations,
}
if country_change_summary is not None:
    outputs["country_pair_change_summary.csv"] = country_change_summary
    outputs["country_pair_transition_counts.csv"] = country_transition_counts
output_directory.mkdir(parents=True, exist_ok=True)
for filename, frame in outputs.items():
    frame.to_csv(output_directory / filename, index=False)
print(f"Saved {len(outputs)} classification outputs to {output_directory}")